# 0. 核心思想

去除 Critic 和 Reward Model，只保留：
- Actor（还是需要训练）：负责学习。
- Ref Model（依旧保持冻结）：负责当参照物（防止学偏）。

DPO 不再给单个答案打分，而是拿出一对答案：
- Chosen (好答案 $y_w$)：win
- Rejected (坏答案 $y_l$)：lose

它的目标非常简单粗暴：让模型生成 $y_w$ 的概率，远远大于生成 $y_l$ 的概率。

# 1. 训练目标

**DPO 损失函数：**

$$
L_{\mathrm{DPO}}(\theta) = -\mathbb{E}_{(x,y_w,y_l)\sim\mathcal{D}} \left[ \log\sigma\left( \beta\left[ \log\frac{\pi_{\theta}(y_w\mid x)}{\pi_{\mathrm{ref}}(y_w\mid x)} - \log\frac{\pi_{\theta}(y_l\mid x)}{\pi_{\mathrm{ref}}(y_l\mid x)} \right] \right) \right]
$$


- $\mathbb{E}_{(x,y_w,y_l)\sim\mathcal{D}}$：在**偏好数据集**上求期望
- $y_w$ ：偏好回答，chosen/winner
- $y_l$ ：非偏好回答，rejected/loser
- $\sigma$ ：Sigmoid 函数，将奖励差映射为 [0, 1] 的概率

KL 散度：

DPO 把奖励和 KL 约束合并为了同一个公式，即：
$$\log \frac{\pi_{\theta}(y_w)}{\pi_{\mathrm{ref}}(y_w)}, \log \frac{\pi_{\theta}(y_l)}{\pi_{\mathrm{ref}}(y_l)}$$
对数概率比同时编码了"什么是好的"和"别偏离太远"。

$\beta$ 参数控制这种约束的强度——$\beta$ 越大，策略越倾向于接近参考模型。


# 2. 补充

**$\beta$ 大小的影响：**

- $\beta$ 是温度参数，控制隐式奖励差的缩放，本质上等价于 PPO 中 KL 惩罚系数的作用
- $\beta$ 过小（如 0.01）：loss 对偏好对的区分度极高，可能导致过拟合训练数据中的噪声偏好，训练不稳定
- $\beta$ 过大（如 1.0 以上）：策略几乎不会偏离参考模型，等于 RL 训练没有生效，策略更新幅度极小。

通常建议 $\beta$∈[0.1,0.5]


**DPO 为什么被认为是离线的？**

DPO 完全依赖离线偏好数据 $(x, y_w, y_l)$，训练过程中不做任何新的采样。

- 适用场景：已经拥有高质量偏好数据的场景（如人类标注的指令遵循偏好对）、对齐任务（alignment）中已有充分的好坏样本对比数据
- 不适用场景：需要模型涌现全新推理策略的任务（如 DeepSeek-R1 的长链推理），此时需要在线 RL（PPO/GRPO）

# 3. Reference

[1] [DPO：直接偏好优化](https://haozhe-xing.github.io/agent_learning/zh/chapter_agentic_rl/04_dpo.html) <br>
[2] [看完能和外婆解释的PPO, DPO, GRPO强化学习](https://zhuanlan.zhihu.com/p/1984387073625593089) <br>